In [1]:
import pandas as pd
import numpy as np
from ax.service.ax_client import AxClient, ObjectiveProperties
import matplotlib.pyplot as plt
from ax.modelbridge.registry import Generators

from ax.core.observation import ObservationFeatures
from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy
import json
import subprocess
import os
import re

/var/folders/hk/7jjnrbrj53n1t8_bmkhf0_k00000gn/T/ipykernel_35583/1730797829.py:8: DeprecationWarning: Please import from 'ax.generation_strategy.generation_strategy' instead of 'ax.modelbridge.generation_strategy'. The latter is deprecated and will be removed in a future release.
  from ax.modelbridge.generation_strategy import GenerationStep, GenerationStrategy


In [10]:
iteration_to_update = 3
optimizer_file_path = 'iteration_' + str(iteration_to_update) + '/optimizer/optimizer_'
ax_to_update_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"



In [ ]:
ax_to_update = AxClient.load_from_json_file(ax_to_update_path)
trials_to_update = ax_to_update.get_trials_data_frame()
trials_to_update


,trial_index,arm_name,trial_status,generation_node,success,surfactant_input,complexity,Drug_MW,Drug_LogP,Drug_TPSA,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc
0,0,0_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2063,0.3073,0.0373,48,60,49,31,96,8,9,35,86,100
1,1,1_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2063,0.3073,0.0373,74,0,88,66,22,52,58,79,35,100
2,2,2_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.4045,0.4196,0.0728,78,88,1,3,32,94,44,58,1,100
3,3,3_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.4045,0.4196,0.0728,0,46,61,94,63,49,98,0,52,100
4,4,4_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2962,0.4364,0.0493,16,82,85,84,71,23,35,65,21,100
5,5,5_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2962,0.4364,0.0493,93,27,27,17,47,67,83,22,71,100
6,6,6_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.3528,0.2810,0.0711,59,65,62,55,7,76,19,43,94,100
7,7,7_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.3528,0.2810,0.0711,31,19,25,46,87,32,74,99,44,100
8,8,8_0,COMPLETED,GenerationStep_1,1.0,1.0,0.750,0.2063,0.3073,0.0373,0,100,100,0,100,100,100,100,100,100
9,9,9_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.2063,0.3073,0.0373,0,100,0,0,0,0,100,100,100,100


In [7]:
def update_data_to_optimizer(ax_client, list_of_new_failures):

    # Load the existing optimizer state
    before_update_path = optimizer_file_path + f"{iteration_to_update}_before_update.json"
    updated_path = optimizer_file_path + f"{iteration_to_update}_loaded.json"

    ax_client.save_to_json_file(before_update_path)

    # Fetch current trials
    trials_df = ax_client.get_trials_data_frame()

    for trial_index in list_of_new_failures:
        # Make sure we actually have this trial
        if trial_index not in trials_df["trial_index"].values:
            print(f"Trial {trial_index} not found – skipping.")
            continue

        # Build the forced-failure payload
        new_data = {
            "success": 0,
            "surfactant_input": 1,
            "complexity": 1,
        }

        # Update the trial in-place
        ax_client.update_trial_data(trial_index=trial_index, raw_data=new_data)

    ax_client.save_to_json_file(updated_path)

    print(f"Updated trial {trial_index}: set success=0, surfactant_input=1, complexity=1")

    return ax_client

In [8]:
new_ax_client = update_data_to_optimizer(ax_to_update, [18,19,21])

[INFO 07-14 13:57:23] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 18.
[INFO 07-14 13:57:23] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 19.
[INFO 07-14 13:57:23] ax.service.ax_client: Added data: {'success': (0.0, None), 'surfactant_input': (1.0, None), 'complexity': (1.0, None)} to trial 21.


Updated trial 21: set success=0, surfactant_input=1, complexity=1


In [9]:
updated_trials = new_ax_client.get_trials_data_frame()
updated_trials

,trial_index,arm_name,trial_status,generation_node,success,surfactant_input,complexity,Drug_MW,Drug_LogP,Drug_TPSA,s1,s2,s3,s4,s5,s6,s7,s8,surfactant_conc,drug_conc
0,0,0_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2063,0.3073,0.0373,48,60,49,31,96,8,9,35,86,100
1,1,1_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2063,0.3073,0.0373,74,0,88,66,22,52,58,79,35,100
2,2,2_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.4045,0.4196,0.0728,78,88,1,3,32,94,44,58,1,100
3,3,3_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.4045,0.4196,0.0728,0,46,61,94,63,49,98,0,52,100
4,4,4_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2962,0.4364,0.0493,16,82,85,84,71,23,35,65,21,100
5,5,5_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.2962,0.4364,0.0493,93,27,27,17,47,67,83,22,71,100
6,6,6_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.3528,0.2810,0.0711,59,65,62,55,7,76,19,43,94,100
7,7,7_0,COMPLETED,GenerationStep_0,0.0,1.0,1.000,0.3528,0.2810,0.0711,31,19,25,46,87,32,74,99,44,100
8,8,8_0,COMPLETED,GenerationStep_1,1.0,1.0,0.750,0.2063,0.3073,0.0373,0,100,100,0,100,100,100,100,100,100
9,9,9_0,COMPLETED,GenerationStep_1,1.0,1.0,0.375,0.2063,0.3073,0.0373,0,100,0,0,0,0,100,100,100,100
